# Modelos no lineales (1): Threshold VAR y el Fenómeno El Niño

Hasta la sesión 3 los coeficientes eran fijos: un choque se propaga igual en 1998 que en 2019, en
un año normal que en medio de un Niño costero. Este notebook rompe ese supuesto con un
**Threshold VAR (TVAR)**: dos VAR, uno por régimen, y una variable observable que decide en qué
régimen está la economía.

La variable que decide es la **anomalía de la temperatura superficial del mar en la región Niño
1+2** (frente a la costa de Perú y Ecuador, NOAA), $F_t$. El régimen cambia cuando la anomalía
rezagada cruza un umbral:

$$
\mathbf{y}_t = \big(c_1 + B_{1,1}\mathbf{y}_{t-1} + B_{1,2}\mathbf{y}_{t-2} + \mathbf{u}_{1,t}\big)\,S_t
 + \big(c_2 + B_{2,1}\mathbf{y}_{t-1} + B_{2,2}\mathbf{y}_{t-2} + \mathbf{u}_{2,t}\big)(1 - S_t),
\qquad \mathbf{u}_{i,t} \sim N(0, \Sigma_i),
$$

$$
S_t = 1 \iff F_{t-d} \le Z^*,
$$

con el umbral $Z^*$ y el rezago $d$ estimados junto con los coeficientes. El sistema es
$\mathbf{y}_t = (F_t,\ y_t,\ \pi_t)'$: la anomalía en grados, el crecimiento interanual del PBI
y la inflación interanual, trimestral, de 1994Q1 a 2026Q2, con la pandemia tratada como en la
sesión 2: escalas de volatilidad de Lenza y Primiceri (2022) para 2020-21.

1. **El régimen**: el umbral, el rezago y los trimestres que el modelo clasifica como Niño, y
   qué cambia con la muestra y con el tratamiento de la pandemia.
2. **Respuestas por régimen**: un calentamiento de 1 °C en tiempos normales y durante un Niño;
   asimetrías de tamaño y de signo.
3. **Pronósticos y escenarios**: el Niño de 2026, que está en curso, con la densidad predictiva
   simulada a mano y comparada con el VAR lineal de la sesión 2.
4. **Varios modelos**: otras variables, más historia, otro índice ENSO y otros tratamientos.

La extensión con volatilidad estocástica, el TVAR-SV de Alessandri y Mumtaz (2019), está en
`MacroPy` como `ThresholdVARSV` y se presenta en `tutorial_thresholdvar.ipynb`; aquí la
volatilidad estocástica llega con el notebook B.

*Tiempo de corrida: uno a dos minutos. El muestreador del TVAR es exacto, no un Gibbs.*

## 1. Datos

`course_data.el_nino` arma el sistema desde la base del curso. Las anomalías ENSO vienen de NOAA
(series `nino12` y `nino34`, promedios trimestrales de la serie mensual, en grados respecto al
promedio de 1981-2010); el PBI y el IPC, del BCRP. La muestra empieza en 1994, cuando la inflación
ya había bajado a dos dígitos después de la hiperinflación, y llega hasta el último dato, 2026Q2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MacroPy import ThresholdVAR, BayesianVAR, generate_series_plot, set_style
from MacroPy.plots_uc import plot_fan_chart
import sys; sys.path.append("..")          # course_data.py vive en la raíz del repositorio
import course_data as cd

set_style()
levels = cd.load()                          # copia congelada de la base del curso
Y = cd.el_nino(levels, end="2026-06-01")    # (nino12, y, pi), 1994Q1-2026Q2
names = list(Y.columns)
titles = ["Anomalía Niño 1+2 (°C)", "PBI (var. % interanual)", "Inflación (var. % interanual)"]
print(f"Muestra: {cd.quarter(Y.index[0])} a {cd.quarter(Y.index[-1])} ({len(Y)} trimestres)")
generate_series_plot(Y, series_titles=titles, title="El Niño costero y la macroeconomía peruana")

In [ ]:
nino = levels["nino12"].dropna().loc["1980-03-01":]
fuertes = nino[nino > 1.5]
print("Trimestres con anomalía mayor a 1.5 °C desde 1980:")
print(", ".join(cd.quarter(d) for d in fuertes.index))
print("\nEl PBI interanual alrededor de la pandemia:")
Y.loc["2019-12-01":"2022-03-01", ["y"]].T.rename(columns=cd.quarter).round(1)

Cinco episodios fuertes en 45 años: 1982-83, 1997-98, 2015-16, 2023 y el que está en curso en
2026. El de 2017, el Niño costero de marzo, fue intenso pero corto y queda por debajo de 1.5 °C en
el promedio trimestral. La muestra contiene tres, 1997-98, 2015-16 y 2023-24; la sección 5
extiende la muestra hasta 1980 para ver si se puede aprovechar el de 1982-83, y la sección 4 usa
el modelo para el de 2026.

Y contiene la pandemia: $-30$ % en 2020Q2 y, por la aritmética de la tasa interanual, $+42$ % un
año después. Son dos observaciones diez veces más grandes que cualquier recesión asociada a un
Niño, y entran también como **rezagos** de los trimestres siguientes. Sin tratamiento, el segundo
régimen del TVAR se dedicaría a ellas (sección 2). El tratamiento es el de la sesión 2:
**escalas de volatilidad de Lenza y Primiceri (2022)**, $s_t \ge 1$ para cada trimestre de
2020Q1 a 2021Q4, que dividen esas filas antes de estimar; la ventana se extiende dos trimestres
más allá del rebote para que los rezagos contaminados también pesen menos. Las escalas se
estiman por verosimilitud marginal en el VAR lineal con el mismo prior, antes de mirar ningún
régimen, y como la tasa interanual pone el segundo pico en 2021Q2 y no hay decaimiento
geométrico que lo describa, las ocho escalas se dejan libres (`covid_n_free=8`).

## 2. El régimen

`ThresholdVAR` tiene la misma interfaz que `BayesianVAR`, incluido el tratamiento de la pandemia.
El prior de los coeficientes es el de Minnesota por observaciones ficticias, uno por régimen, y el
umbral tiene un prior normal centrado en la media muestral de la anomalía. La posterior del umbral
y del rezago se calcula de forma **exacta**: dado el corte de la muestra en dos regímenes, el prior
es conjugado y la verosimilitud marginal de cada corte tiene forma cerrada, así que el muestreador
evalúa todos los cortes admisibles y sortea entre ellos. No hay cadena que pueda quedarse atascada
en un modo.

In [ ]:
window = ("2020Q1", "2021Q4")
model = ThresholdVAR(Y, threshold_var="nino12", lags=2, max_delay=4, post_draws=4000, burnin=0.5,
                     hor=12, irf_1std=0, seed=1,
                     covid_mode="lenza-primiceri", covid_window=window, covid_n_free=8)
print("Escalas de volatilidad estimadas, s_t (las filas se dividen por s_t):")
display(pd.DataFrame({cd.quarter(d): [s] for d, s in zip(model.covid_dates, model.covid_scales)},
                     index=["s_t"]).round(1))
model.model_summary()
draws = model.sample_posterior()

Las escalas dicen dónde está la pandemia en estos datos: 2020Q2 (la caída), 2021Q2 (el rebote de
la tasa interanual) y, con menos fuerza, los trimestres en los que esas dos observaciones son
rezagos, 2020Q3 y 2021Q3. En 2020Q4 y 2021Q1 la escala es uno: nada que tratar.

In [ ]:
display(model.threshold_summary().round(2))
sp = model.split_posterior
print("Posterior del rezago d:", sp.groupby("delay")["probability"].sum().round(2).to_dict())
print("Los cinco cortes más probables (n_regime1 = trimestres en el régimen normal):")
sp.sort_values("probability", ascending=False).head(5)[
    ["delay", "n_regime1", "lower", "upper", "probability"]].round(3)

El umbral está cerca de **1.2 °C** y el rezago es de **tres trimestres**: la economía entra al
régimen Niño casi un año después de que el mar se calienta. Es el tiempo que tardan en
materializarse las lluvias, las pérdidas agrícolas y pesqueras y el daño a la infraestructura. La
posterior se concentra en unos pocos cortes de la muestra, con unos trece trimestres en el régimen
Niño.

In [ ]:
model.plot_regimes(regime_label="Régimen Niño", prob_label="P(Niño)",
                   var_label="Anomalía Niño 1+2 (°C)", title="Regímenes El Niño, 1994-2026")
p_nino = model.regime_probability()
print("Trimestres en régimen Niño (P > 0.5):", ", ".join(cd.quarter(d) for d in p_nino.index[p_nino > 0.5]))

Tres episodios: **1998Q1 a 1999Q1**, el año posterior al Niño extraordinario de 1997-98; **2016**,
el año posterior al de 2015-16 (más 2017Q4); y **2024Q1 a 2024Q3**, el año posterior al Niño
costero de 2023. La línea punteada es el umbral; la curva azul, la probabilidad posterior del
régimen Niño, alta en esos trimestres y casi cero en el resto. El régimen no coincide con el pico
de la anomalía sino con el año siguiente: eso es el rezago $d$.

¿Vale la pena el segundo régimen? `log_marginal_likelihood` compara el TVAR con el VAR lineal con
el mismo prior y el mismo tratamiento:

In [ ]:
model.log_marginal_likelihood().round(1)

El factor de Bayes es **ligeramente desfavorable** al TVAR. Es una lección y no un detalle: con
trece trimestres en el régimen Niño y siete coeficientes por ecuación, el segundo VAR descansa
mucho sobre el prior, y la verosimilitud marginal cobra ese precio. Los papers de la sesión usan
datos mensuales con varios cientos de observaciones.

**¿Qué cambia con la muestra y con el tratamiento?** El mismo modelo estimado hasta 2019 (sin
pandemia que tratar), con la muestra completa sin tratamiento, y con la alternativa al
escalamiento: sacar de la verosimilitud los trimestres 2020Q1 a 2021Q2 y los dos siguientes, cuyos
rezagos son la caída y el rebote (`covid_mode="dummies"`, una dummy por observación con prior
difuso). Para cada uno, el umbral, el rezago, los trimestres Niño, el factor de Bayes y la
respuesta acumulada del PBI a $+1$ °C en cada régimen, que la sección 3 explica.

In [ ]:
def episodes(index):
    '''Trimestres consecutivos como rangos: 1998Q1-1999Q1, 2016Q1-2016Q4.'''
    out, start, prev = [], None, None
    for d in index:
        if start is None:
            start = prev = d
        elif (d.year - prev.year) * 4 + (d.quarter - prev.quarter) == 1:
            prev = d
        else:
            out.append((start, prev)); start = prev = d
    if start is not None:
        out.append((start, prev))
    return ", ".join(cd.quarter(a) if a == b else f"{cd.quarter(a)}-{cd.quarter(b)}" for a, b in out)

def regime_row(m, irf):
    ts, p2, sp = m.threshold_summary(), m.regime_probability(), m.split_posterior
    yi = m.names.index("y")
    cum = {reg: np.median(irf[reg]["level"][:, :9, yi, 0], axis=0).sum() if irf[reg] else np.nan
           for reg in ("regime1", "regime2")}
    return {"T": m.T, "Z*": ts.loc["threshold (Z*)", "p50"], "p5": ts.loc["threshold (Z*)", "p5"],
            "p95": ts.loc["threshold (Z*)", "p95"], "d": int(sp.groupby("delay")["probability"].sum().idxmax()),
            "trimestres Niño": int((p2 > 0.5).sum()), "episodios": episodes(p2.index[p2 > 0.5]),
            "log BF": m.log_marginal_likelihood()["log Bayes factor"],
            "PBI acum. 8, normal": cum["regime1"], "PBI acum. 8, Niño": cum["regime2"]}

def fit(data, **kw):
    m = ThresholdVAR(data, threshold_var="nino12", lags=2, max_delay=4, post_draws=4000, burnin=0.5,
                     hor=12, irf_1std=0, seed=1, **kw)
    m.sample_posterior(show_progress=False)
    return m, m.compute_irfs(shock="level", n_draws=100, reps=20, scale=1.0, show_progress=False)

tvar19, irf19 = fit(Y.loc[:"2019-12-01"])
raw26, irf_raw = fit(Y)
dum26, irf_dum = fit(Y, covid_mode="dummies", covid_window=("2020Q1", "2021Q2"))
irf1 = model.compute_irfs(shock="level", n_draws=200, reps=25, scale=1.0)
rows = {"1994Q1-2019Q4, sin la pandemia": regime_row(tvar19, irf19),
        "1994Q1-2026Q2, sin tratamiento": regime_row(raw26, irf_raw),
        "1994Q1-2026Q2, dummies 2020Q1-2021Q2 (+ rezagos)": regime_row(dum26, irf_dum),
        "1994Q1-2026Q2, Lenza-Primiceri 2020Q1-2021Q4 (base)": regime_row(model, irf1)}
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(rows).T.round(2)

Tres lecturas. Hasta 2019 el modelo es el mismo: umbral de 1.3 °C, rezago de tres trimestres,
1998-99 y 2016, y una respuesta en el régimen Niño tres veces mayor que en el normal; la muestra
completa agrega el episodio de 2024 y reduce el costo del segundo régimen (el factor de Bayes sube
de $-4$ a $-1$). Sin tratamiento, el "régimen Niño" pasa a ser el régimen de varianza alta: el
umbral cae a cero, 59 trimestres caen en él, el factor de Bayes se dispara y la asimetría
desaparece. **Un factor de Bayes alto no garantiza que el régimen sea el que uno cree.** Y las dos
formas de tratar la pandemia, ponderar o excluir, llegan al mismo sitio: cuando las escalas son
grandes donde hace falta, dividir una fila por diez es casi lo mismo que borrarla. Se usa la de
Lenza y Primiceri porque mantiene las observaciones y tiene una justificación en la literatura;
la sección 5 vuelve sobre la ventana.

## 3. Respuestas por régimen

En un modelo no lineal la impulso-respuesta depende de la historia, del régimen y del tamaño y
signo del choque. `compute_irfs` calcula **respuestas generalizadas** (Koop, Pesaran y Potter,
1996): para cada extracción de la posterior simula la economía con y sin el choque, promediando
sobre las historias de cada régimen, y deja que el régimen cambie por el camino. Con `irf_1std=0`
el choque es de **una unidad**: un calentamiento de 1 °C en la región Niño 1+2.

In [ ]:
model.plot_irfs(shock="nino12", response_vars=["y", "pi"], cred=0.68,
                regime_labels=("Normal", "Niño"), series_titles=titles[1:],
                title="Respuesta a un calentamiento de 1 °C en la región Niño 1+2")

In [ ]:
def response_table(irf, shock, var, horizons=(0, 2, 4, 8)):
    s, v = names.index(shock), names.index(var)
    rows = {}
    for reg, label in (("regime1", "normal"), ("regime2", "Niño")):
        med = np.median(irf[reg]["level"][:, :, v, s], axis=0)
        rows[label] = {**{f"h = {h}": med[h] for h in horizons}, "acumulado a 8": med[:9].sum()}
    return pd.DataFrame(rows).T

print("Respuesta mediana a +1 °C, en puntos porcentuales")
pd.concat({"PBI": response_table(irf1, "nino12", "y"),
           "inflación": response_table(irf1, "nino12", "pi")}).round(2)

La asimetría es grande y está **al impacto**. En tiempos normales un grado más de temperatura
casi no mueve el PBI en el trimestre y le resta unas décimas de crecimiento al año siguiente;
durante un Niño el mismo grado resta cerca de dos puntos de crecimiento de inmediato y el efecto
acumulado en dos años triplica al del régimen normal, con bandas que no se cruzan. Sobre la
inflación el efecto es pequeño y transitorio en tiempos normales, y de cuatro décimas al impacto
durante un Niño (alimentos y transporte), que se disipa en un año.

**Tamaño y signo.** El modelo es lineal dentro de cada régimen, así que un choque de 3 °C es tres
veces el de 1 °C mientras la economía no cambie de régimen. Lo que puede cambiar el régimen es el
propio choque: un calentamiento grande en tiempos normales cruza el umbral y, tres trimestres
después, activa la dinámica Niño. Un enfriamiento (La Niña) es el espejo del calentamiento dentro
del régimen.

In [ ]:
irf3 = model.compute_irfs(shock="level", n_draws=200, reps=25, scale=3.0)
irfm = model.compute_irfs(shock="level", n_draws=200, reps=25, scale=-1.0)
size = pd.DataFrame({
    "+1 °C": response_table(irf1, "nino12", "y")["acumulado a 8"],
    "+3 °C": response_table(irf3, "nino12", "y")["acumulado a 8"],
    "-1 °C (La Niña)": response_table(irfm, "nino12", "y")["acumulado a 8"]})
size["(+3) / (+1)"] = size["+3 °C"] / size["+1 °C"]
print("PBI, respuesta acumulada a 8 trimestres")
size.round(2)

En el régimen Niño la razón es tres, como en un modelo lineal. En el régimen normal es mayor:
parte de las simulaciones con +3 °C terminan en el régimen Niño. La no linealidad de tamaño es
modesta porque el rezago de tres trimestres deja el régimen inicial en pie durante buena parte del
horizonte en que el choque importa.

## 4. Pronósticos y escenarios: el Niño de 2026

En octubre de 2026 la anomalía Niño 1+2 ya superó los 3 °C en el promedio del tercer trimestre.
Un modelo con regímenes responde a una pregunta que el VAR lineal no puede: **¿cuándo y cuánto**
se sentirá en el PBI y en la inflación? La densidad predictiva se simula **a mano**, como en la
sesión 2, con una diferencia: en cada paso la simulación consulta la anomalía rezagada para
decidir con qué régimen avanza.

Las condiciones iniciales son las observadas hasta 2026Q2 (el último dato macro) y la anomalía de
2026Q3, ya conocida. Los parámetros son los del modelo base, estimado con toda la muestra, así que
el Niño de 2023-24 ya está en el régimen.

In [ ]:
def simulate(parameters, history, h, threshold_var, imposed=None, seed=0):
    '''Sendas futuras de un VAR con dos regimenes. `parameters` es una lista, una entrada por
    extraccion: {"B": (B1, B2), "C": (chol1, chol2), "tar": Z*, "delay": d}, con B de k x n y
    filas [rezago 1, rezago 2, constante]. `imposed` fija variables por sus propios choques
    estructurales (orden recursivo), como un pronostico condicional; NaN deja el paso libre.'''
    rng = np.random.default_rng(seed)
    names = list(history.columns)
    n, L = len(names), 2
    thr = names.index(threshold_var)
    fixed = sorted((names.index(v), np.asarray(path, float)) for v, path in (imposed or {}).items())
    pre = history.shape[0]
    paths = np.empty((len(parameters), h, n))
    regime = np.empty((len(parameters), h), dtype=int)
    for m, par in enumerate(parameters):
        path = np.vstack([history.to_numpy(), np.zeros((h, n))])
        for j in range(h):
            t = pre + j
            s = 0 if path[t - par["delay"], thr] <= par["tar"] else 1       # regimen activo
            lags = np.concatenate([path[t - l] for l in range(1, L + 1)])
            mean = lags @ par["B"][s][:-1] + par["B"][s][-1]
            eps = rng.standard_normal(n)
            Lc = par["C"][s]
            for k, target in fixed:                                       # choque que cumple la senda
                if not np.isnan(target[j]):
                    eps[k] = (target[j] - mean[k] - Lc[k, :k] @ eps[:k]) / Lc[k, k]
            path[t] = mean + Lc @ eps
            regime[m, j] = s
        paths[m] = path[pre:]
    return paths, regime

def tvar_parameters(model, n_draws=1500):
    d = model.draws
    sel = np.linspace(0, d["tar"].shape[0] - 1, n_draws).astype(int)
    return [{"B": (d["B1"][i], d["B2"][i]),
             "C": (np.linalg.cholesky(d["S1"][i]), np.linalg.cholesky(d["S2"][i])),
             "tar": d["tar"][i], "delay": int(d["delay"][i])} for i in sel]

h = 8
history = Y.iloc[-4:]                                           # cuatro trimestres: rezagos y delay
dates = pd.date_range(history.index[-1] + pd.DateOffset(months=3), periods=h, freq="3MS")
nino_2026Q3 = levels.loc["2026-09-01", "nino12"]
print("Condiciones iniciales:"); display(history.round(2))
print(f"Anomalía observada en 2026Q3: {nino_2026Q3:.2f} °C; horizonte {cd.quarter(dates[0])} a {cd.quarter(dates[-1])}")

Cuatro escenarios para la anomalía desde 2026Q4. El primero copia la trayectoria del Niño de
1997-98 desde su pico, el episodio más parecido al actual; el segundo supone una disipación rápida;
el tercero no impone nada más y deja que el propio modelo proyecte la anomalía. El cuarto es un
**contrafactual**: el mar nunca se calentó (anomalía cero desde 2026Q1), y sirve para medir el costo
del Niño como la diferencia con los otros tres.

In [ ]:
como_1998 = levels["nino12"].loc["1997-12-01":"1999-06-01"].to_numpy()      # 1997Q4 a 1999Q2
libre = np.full(h - 1, np.nan)
scenarios = {"como 1997-98": np.r_[nino_2026Q3, como_1998[:h - 1]],
             "disipación rápida": np.r_[nino_2026Q3, 2.0, 0.8, 0.0, 0.0, 0.0, 0.0, 0.0],
             "sin condicionar": np.r_[nino_2026Q3, libre],
             "sin Niño (contrafactual)": np.zeros(h)}
history_cf = history.copy()
history_cf.loc["2026-03-01":, "nino12"] = 0.0           # como si el mar no se hubiera calentado
pars = tvar_parameters(model)
paths, regimes = {}, {}
for name, path in scenarios.items():
    hist = history_cf if name.startswith("sin Niño") else history
    paths[name], regimes[name] = simulate(pars, hist, h, "nino12", {"nino12": path})

labels = [cd.quarter(d) for d in dates]
print("Senda impuesta de la anomalía (°C):")
display(pd.DataFrame(scenarios, index=labels).T.round(2))
print("Probabilidad del régimen Niño en cada trimestre del horizonte:")
pd.DataFrame({k: v.mean(axis=0) for k, v in regimes.items()}, index=labels).T.round(2)

El régimen de 2027 ya está decidido por lo observado: con un rezago de tres trimestres, las
anomalías de 2026Q2 y 2026Q3 ponen a la economía en el régimen Niño desde el primer trimestre de
2027 en cualquier escenario con el calentamiento. Los escenarios difieren en cuánto dura.

In [ ]:
colors = {"como 1997-98": "#B0413E", "disipación rápida": "#E78033", "sin condicionar": "#1F3D5C",
          "sin Niño (contrafactual)": "#7F7F7F"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, var, title in zip(axes, ["y", "pi"], titles[1:]):
    k = names.index(var)
    plot_fan_chart(Y[var], paths["como 1997-98"][:, :, k], dates, bands=(0.90, 0.68),
                   history_from="2022-03-01", hline=0, ax=ax, color=colors["como 1997-98"],
                   center_label="mediana, como 1997-98", history_label="observado",
                   title=f"{title}: Niño de 2026")
    for name in ("disipación rápida", "sin condicionar", "sin Niño (contrafactual)"):
        ax.plot(dates, np.median(paths[name][:, :, k], axis=0), color=colors[name], lw=1.8,
                ls="--" if name.startswith("sin Niño") else "-", label=f"mediana, {name}")
    ax.legend(fontsize=8, loc="lower left")
plt.show()

In [ ]:
def yearly(paths_dict, var, year):
    k = names.index(var)
    cols = [j for j, d in enumerate(dates) if d.year == year]
    return {name: np.percentile(p[:, cols, k].mean(axis=1), [16, 50, 84]) for name, p in paths_dict.items()}

summary = {}
for var, label in (("y", "PBI 2027"), ("pi", "inflación 2027")):
    for name, q in yearly(paths, var, 2027).items():
        summary[(label, name)] = {"p16": q[0], "mediana": q[1], "p84": q[2]}
print("Promedio de 2027, en var. % interanual (percentiles de las trayectorias)")
pd.DataFrame(summary).T.round(1)

Un Niño que repita 1997-98 deja el crecimiento de 2027 en 1.2 %, contra 4.1 % en el
contrafactual: **casi tres puntos de PBI**, con el golpe concentrado en la primera mitad del año
(la mediana cae a −1.2 % en 2027Q2) y una recuperación rápida después. Si el evento se disipa
pronto, el costo baja a poco más de medio punto. Sobre la inflación de 2027 el efecto neto es
pequeño y negativo, 3.0 % contra 3.6 % sin Niño: el impulso inicial de los alimentos se compensa
con la recesión del régimen, y las bandas son anchas.

**El VAR lineal, para comparar.** Un `BayesianVAR` con los mismos datos, rezagos, prior y
tratamiento de la pandemia no conoce regímenes: para él, el Niño es un choque grande a una
variable y nada más. Se simula con la misma función (un solo régimen, umbral infinito) y las
mismas sendas de la anomalía.

In [ ]:
bvar = BayesianVAR(Y, lags=2, prior_type=1, post_draws=4000, burnin=0.5, seed=1, progress=False,
                   covid_mode="lenza-primiceri", covid_window=window, covid_n_free=8)
bvar.sample_posterior()

def linear_parameters(bvar, n_draws=1500):
    beta, sigma = np.asarray(bvar.beta_draws), np.asarray(bvar.Sigma_draws)
    n, k = sigma.shape[1], bvar.ncoeff_eq
    sel = np.linspace(0, beta.shape[0] - 1, n_draws).astype(int)
    out = []
    for i in sel:
        coef = beta[i].reshape(n, k)                 # fila: ecuacion, constante al final
        B = np.vstack([coef[:, :-1].T, coef[:, -1]])
        C = np.linalg.cholesky(sigma[i])
        out.append({"B": (B, B), "C": (C, C), "tar": np.inf, "delay": 1})
    return out

pars_lin = linear_parameters(bvar)
paths_lin = {name: simulate(pars_lin, history_cf if name.startswith("sin Niño") else history,
                            h, "nino12", {"nino12": path})[0] for name, path in scenarios.items()}
y2027 = pd.DataFrame({"TVAR": {k: v[1] for k, v in yearly(paths, "y", 2027).items()},
                      "VAR lineal": {k: v[1] for k, v in yearly(paths_lin, "y", 2027).items()}})
costo = (y2027.loc["sin Niño (contrafactual)"] - y2027).drop("sin Niño (contrafactual)")
costo = costo.rename(columns=lambda c: f"costo del Niño, {c}")
print("PBI 2027: mediana del promedio anual y costo del Niño respecto al contrafactual (pp)")
display(pd.concat([y2027, costo], axis=1).round(1))
k = names.index("y")
print("Trimestre a trimestre, escenario como 1997-98 (mediana del crecimiento del PBI)")
pd.DataFrame({"TVAR": np.median(paths["como 1997-98"][:, :, k], axis=0),
              "VAR lineal": np.median(paths_lin["como 1997-98"][:, :, k], axis=0)}, index=labels).T.round(1)

Los dos modelos coinciden en el costo mediano de un Niño como el de 1997-98, 2.8 puntos de
crecimiento en 2027 en el TVAR y 2.7 en el VAR lineal, pero no en su forma. El TVAR lo concentra:
el régimen Niño golpea en la primera mitad de 2027 (−1.2 % interanual en 2027Q2) y la economía se
recupera en cuanto sale del régimen; el VAR lineal reparte un golpe menor, cerca de 1 % de
crecimiento durante todo 2027, y lo prolonga. Por eso difieren cuando el evento se disipa rápido:
el TVAR sale del régimen en 2027Q4 y el costo cae a medio punto, mientras que el VAR lineal sigue
arrastrando el choque (1.3 puntos). Y las dos conclusiones hay que tomarlas con pinzas: el régimen
Niño está estimado con trece trimestres, y el modelo no sabe nada de la respuesta fiscal ni de la
reconstrucción que siguió a cada Niño.

## 5. Varios modelos

El régimen y la asimetría, ¿dependen de la elección del sistema? Se reestima el TVAR, siempre con
las escalas de Lenza y Primiceri, con controles externos (el precio del cobre y el PBI de Estados
Unidos, para que el régimen Niño no absorba la crisis asiática de 1998 ni el fin del superciclo
en 2016), con otras variables domésticas (la inversión privada y la tasa de interés de corto
plazo, empalmada con la de los certificados de depósito del BCRP para llegar a 1994), con más
historia, con el índice Niño 3.4 en lugar del 1+2, y con otras ventanas para las escalas. Para
cada uno: el umbral, el rezago, los trimestres Niño, el factor de Bayes contra el VAR lineal y la
respuesta acumulada del PBI a +1 °C en cada régimen.

In [ ]:
g = cd.yoy(levels[["pbi", "invpriv", "ipc"]]).rename(columns={"pbi": "y", "invpriv": "inv", "ipc": "pi"})
base = levels[["nino12", "nino34"]].join(g) \
    .join(cd.yoy(levels[["cobre", "usgdp"]]).rename(columns={"cobre": "cu", "usgdp": "yus"})) \
    .join(cd.interbank_long(levels))                    # tasa empalmada con los CD del BCRP desde 1992

def fit_spec(cols, threshold, start, end, window=("2020Q1", "2021Q4"), n_free=8):
    df = base[cols].loc[start:end].dropna()
    m = ThresholdVAR(df, threshold_var=threshold, lags=2, max_delay=4, post_draws=4000,
                     burnin=0.5, hor=12, irf_1std=0, seed=1,
                     covid_mode="lenza-primiceri", covid_window=window, covid_n_free=n_free)
    m.sample_posterior(show_progress=False)
    irf = m.compute_irfs(shock="level", n_draws=100, reps=20, scale=1.0, show_progress=False)
    row = regime_row(m, irf)
    row["muestra"] = f"{cd.quarter(df.index[0])}-{cd.quarter(df.index[-1])}"
    return row

specs = {
    "base: y, pi":                   (["nino12", "y", "pi"], "nino12", "1994-03-01", "2026-06-01"),
    "control: precio del cobre":     (["nino12", "cu", "y", "pi"], "nino12", "1994-03-01", "2026-06-01"),
    "control: PBI de EE.UU.":        (["nino12", "yus", "y", "pi"], "nino12", "1994-03-01", "2026-06-01"),
    "con inversión privada":         (["nino12", "y", "inv", "pi"], "nino12", "1994-03-01", "2026-06-01"),
    "con tasa de interés":           (["nino12", "y", "pi", "i"], "nino12", "1994-03-01", "2026-06-01"),
    "desde 1992":                    (["nino12", "y", "pi"], "nino12", "1992-03-01", "2026-06-01"),
    "umbral Niño 3.4":               (["nino34", "y", "pi"], "nino34", "1994-03-01", "2026-06-01"),
    "desde 1980: y, inversión":      (["nino12", "y", "inv"], "nino12", "1980-03-01", "2026-06-01"),
    "desde 1980: solo PBI":          (["nino12", "y"], "nino12", "1980-03-01", "2026-06-01"),
    "ventana corta 2020Q1-2021Q2":   (["nino12", "y", "pi"], "nino12", "1994-03-01", "2026-06-01", ("2020Q1", "2021Q2"), 6),
    "ventana larga 2020Q1-2022Q2":   (["nino12", "y", "pi"], "nino12", "1994-03-01", "2026-06-01", ("2020Q1", "2022Q2"), 10),
    "3 escalas libres y decaimiento": (["nino12", "y", "pi"], "nino12", "1994-03-01", "2026-06-01", ("2020Q1", "2021Q4"), 3),
}
table = pd.DataFrame({label: fit_spec(*args) for label, args in specs.items()}).T
table = table[["muestra", "T", "Z*", "d", "trimestres Niño", "episodios", "log BF", "PBI acum. 8, normal", "PBI acum. 8, Niño"]]
table.round(2)

Cuatro lecturas.

* **El régimen es robusto** a las variables del sistema: con los controles externos, con
  inversión, con la tasa de interés o con más historia desde 1992, el rezago es de tres
  trimestres y los episodios son los mismos, 1998-99, 2016 y 2024, con el umbral entre 1.2 y 1.4
  °C (1.8 con la tasa). La respuesta acumulada del PBI en el régimen Niño es tres veces la del
  normal en todos los casos (de −8 a −9 contra −2 a −4), así que la asimetría no es la crisis
  asiática ni el fin del superciclo disfrazados de Niño.
* **El factor de Bayes es modesto y de signo cambiante**: negativo en el sistema base y con los
  controles, positivo con inversión, con la tasa, desde 1992 y desde 1980. El apoyo de 1992 no
  viene solo del Niño: la inflación interanual todavía era de 50 a 100 %, y 1993Q1 entra al
  régimen.
* **Es la región 1+2, no la 3.4.** Con el índice Niño 3.4, el del Pacífico central que define
  los eventos globales, el umbral mezcla 1998 y 2016 con 2003Q4 y 2010-11, y la asimetría casi
  desaparece (−2.4 contra −3.5): lo que afecta a Perú es el calentamiento frente a su costa.
* **Más historia sí suma, con cuidado.** Desde 1980 el modelo recupera el Niño de 1982-83
  (1983Q3 a 1984Q2) junto con 1998, 2016 y 2024, con 18 trimestres en el régimen y la misma
  asimetría; la inflación no puede entrar (cuatro dígitos en 1988-90), así que el sistema se
  reduce al PBI, solo o con la inversión. Y la ventana de las escalas importa: si termina en
  2021Q2 y deja los rezagos contaminados sin ponderar, el umbral baja a 1.1 °C y la asimetría se
  reduce a la mitad; extenderla a 2022Q2 o imponer el decaimiento geométrico de Lenza y Primiceri
  (tres escalas libres) no cambia nada. Es la razón de la ventana 2020Q1-2021Q4.

## Ideas centrales

* **El régimen es el año siguiente al calentamiento.** El umbral está cerca de 1.2 °C y el rezago es
  de tres trimestres: trece trimestres caen en el régimen Niño (1998-99, 2016, 2017Q4 y
  2024Q1-Q3). El rezago es una regularidad de esta muestra, no un plazo físico garantizado.
* **La evidencia formal es limitada.** El log factor de Bayes frente al VAR lineal es $-1.1$: con
  pocos trimestres en el segundo régimen, el prior pesa. Y un factor de Bayes alto tampoco basta:
  sin tratar la pandemia, el segundo régimen pasa a ser el de varianza alta.
* **La asimetría está al impacto.** Un calentamiento de 1 °C resta 0.2 puntos de crecimiento en
  tiempos normales y 1.8 durante un Niño; a cuatro trimestres, 0.5 y 0.9. El tamaño y el signo del
  choque pueden cambiar el régimen futuro.
* **Los escenarios dimensionan riesgos.** Un Niño como el de 1997-98 deja el crecimiento de 2027
  en 1.2 %, unos tres puntos menos que el contrafactual sin Niño (4.1 %). Son escenarios
  condicionales a trayectorias impuestas de la temperatura, y el régimen puede cambiar en cada
  paso: el TVAR concentra el golpe en la primera mitad de 2027, el VAR lineal lo reparte.


## Ejercicios

1. **El Niño de 1982-83 fuera de muestra.** Con el modelo base, simule desde 1982Q3 imponiendo la
   senda observada de la anomalía hasta 1984Q2 (solo necesita el PBI como historia: use el sistema
   `(nino12, y)`). Compare la mediana con el PBI observado en 1983, que cayó 10 %. ¿Qué parte de la
   caída atribuye el modelo al Niño y qué parte queda para la crisis de la deuda?
2. **Umbral exógeno.** Sánchez y Vassallo (2026) fijan el umbral de confianza en 50 puntos en
   lugar de estimarlo. Fije `threshold_prior_mean=1.0` con `prior_params={"tarvariance": 0.01}`,
   el límite del Niño moderado en el índice ICEN del ENFEN, y compare los regímenes y las
   respuestas con los de la sección 2. ¿Y con 0.4 °C, el límite del Niño débil?
3. **Escenarios alternativos.** Repita la sección 4 con la senda del Niño de 2015-16 y con la de
   2023. ¿En qué trimestre de 2027 cae más el crecimiento en cada caso y por qué?
4. **Choques financieros.** Reemplace la anomalía por el VIX como variable umbral en el sistema
   macrofiscal `(vix, x, y, r)` de las sesiones 1 y 2. ¿Aparecen regímenes de estrés financiero
   global? ¿Qué dice el factor de Bayes?

## Referencias

- Aguirre, J., Ledesma, A., Pérez, F. y Rojas, Y. (2025). El Niño Costero y la economía peruana: efectos sobre actividad e inflación. Banco Central de Reserva del Perú, Documento de Trabajo 2025-008.
- Alessandri, P. y Mumtaz, H. (2017). Financial conditions and density forecasts for US output and inflation. *Review of Economic Dynamics* 24, 66-78.
- Alessandri, P. y Mumtaz, H. (2019). Financial regimes and uncertainty shocks. *Journal of Monetary Economics* 101, 31-46.
- Chen, C.W.S. y Lee, J.C. (1995). Bayesian inference of threshold autoregressive models. *Journal of Time Series Analysis* 16(5), 483-492.
- Koop, G., Pesaran, M.H. y Potter, S.M. (1996). Impulse response analysis in nonlinear multivariate models. *Journal of Econometrics* 74(1), 119-147.
- Lenza, M. y Primiceri, G.E. (2022). How to estimate a vector autoregression after March 2020. *Journal of Applied Econometrics* 37(4), 688-699.
- Sánchez, W. y Vassallo, R. (2026). Efectos asimétricos de choques externos en la economía peruana: el rol de la confianza. Consejo Fiscal del Perú, Documento de Investigación 01-2026.
- NOAA Physical Sciences Laboratory. Niño 1+2 and Niño 3.4 SST anomalies (ERSST v6), https://psl.noaa.gov/data/climateindices/.